---
title: "Synthetic identifier reuse and network context"
execute:
  freeze: auto
---

This notebook generates the synthetic users and events used by the identifier-reuse methodology experiment, validates their observation windows, and demonstrates a small **authored synthetic** network-context fixture. It does not query real IP services and it does not evaluate public-list accuracy.

The notebook stops at data generation and identifier-reuse summaries. The clustering and sensitivity analyses used by the site page live in the separate experiment scripts documented in `experiments/identifier-synchrony/README.md`.

In [1]:
#| label: setup
from pathlib import Path

import numpy as np
import pandas as pd

from openbotrisk.experiments.identifier_synchrony.generate_data import (
    HORIZON_DAYS,
    SEED,
    T0,
    _rand_ts_organic,
    authored_network_fixture,
    generate_synthetic_data,
)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GENERATED = REPO / "experiments/identifier-synchrony/generated"

## Generate and validate the scenario

Generation is deterministic for a fixed seed. Every event must fall on or after that user's registration and strictly before the observation horizon. Campaign schedules that cannot fit a custom horizon raise an error rather than being clamped.

In [2]:
#| label: generate-and-check
users, events = generate_synthetic_data(seed=SEED, start=T0, horizon_days=HORIZON_DAYS)
observation_end = T0 + pd.Timedelta(days=HORIZON_DAYS)
event_windows = events.merge(
    users[["user_id", "group", "reg_ts"]],
    on="user_id",
    validate="many_to_one",
)
timing_check = event_windows.groupby("group").apply(
    lambda rows: pd.Series(
        {
            "events": len(rows),
            "before_registration": (rows["ts"] < rows["reg_ts"]).sum(),
            "at_or_after_observation_end": (rows["ts"] >= observation_end).sum(),
        }
    ),
    include_groups=False,
)
timing_check

,events,before_registration,at_or_after_observation_end
group,,,
legit,20303,0,0
ring_careful,1902,0,0
ring_lazy,1891,0,0
ring_mid,654,0,0
travel_agent,562,0,0


In [3]:
#| label: daytime-pattern
organic_sample = _rand_ts_organic(np.random.default_rng(123), 20_000)
sample_hour = np.asarray(organic_sample.hour) + np.asarray(organic_sample.minute) / 60
pd.Series(
    {
        "08:00-22:00 share": ((sample_hour >= 8) & (sample_hour < 22)).mean(),
        "sample size": len(organic_sample),
    }
).round(3)

08:00-22:00 share        0.839
sample size          20000.000
dtype: float64

## Authored synthetic network context

Identifiers such as `ip_res_000607` are scenario keys, not routable addresses. The following fixture is authored specifically for this synthetic example. It is separate from the public-data demonstration in `public_ip_enrichment_demo.ipynb`, contains no measured claims, and is constructed without reading the ground-truth `group` column.

In [4]:
#| label: synthetic-network-fixture
network_fixture = authored_network_fixture()
network_fixture

,ip,network_context,datacentre_listed,tor_exit_listed,fixture_source
0,ip_cgnat_000601,university shared egress,False,False,authored synthetic scenario
1,ip_office_000602,business office egress,False,False,authored synthetic scenario
2,ip_dc_000603,scenario hosting range,True,False,authored synthetic scenario
3,ip_dc_000604,scenario hosting range,True,True,authored synthetic scenario
4,ip_res_000607,scenario residential access,False,False,authored synthetic scenario
5,ip_res_000657,scenario mobile access,False,False,authored synthetic scenario


In [5]:
#| label: synthetic-enrichment-join
# `group` is retained below only to audit the authored scenario; it is not a feature.
synthetic_context = users.merge(
    network_fixture, on="ip", how="inner", validate="many_to_one"
)
context_summary = (
    synthetic_context.groupby(
        [
            "ip",
            "network_context",
            "datacentre_listed",
            "tor_exit_listed",
            "group",
        ],
        observed=True,
    )
    .size()
    .rename("users")
    .reset_index()
)
context_summary

,ip,network_context,datacentre_listed,tor_exit_listed,group,users
0,ip_cgnat_000601,university shared egress,False,False,legit,120
1,ip_dc_000603,scenario hosting range,True,False,ring_lazy,20
2,ip_dc_000604,scenario hosting range,True,True,ring_lazy,10
3,ip_office_000602,business office egress,False,False,travel_agent,25
4,ip_res_000607,scenario residential access,False,False,ring_careful,1
5,ip_res_000657,scenario mobile access,False,False,ring_mid,1


The university and office identifiers show benign users sharing infrastructure. The careful and mixed-ring examples use authored residential/mobile context with both flags false, showing that suspicious scenario activity need not occur on hosting or Tor infrastructure. Conversely, a hosting or Tor flag describes the scenario network; it is not an automatic abuse verdict. No fixture field is derived from `group`, and `group` is not used as model input.

## Identifier-reuse summary

These descriptive tables show how the scenario was constructed. They are not a synchrony evaluation.

In [6]:
#| label: reuse-summary
identifier_columns = ["card", "email", "phone", "ip", "asn", "device_id"]
reuse_summary = []
for group, rows in users.groupby("group", sort=False):
    for column in identifier_columns:
        counts = rows[column].value_counts()
        reuse_summary.append(
            {
                "group": group,
                "identifier": column,
                "unique_values": counts.size,
                "maximum_users_per_value": counts.max(),
            }
        )
pd.DataFrame(reuse_summary).pivot(
    index="group", columns="identifier", values="maximum_users_per_value"
)

identifier,asn,card,device_id,email,ip,phone
group,,,,,,
legit,219,2,1,1,120,1
ring_careful,19,1,1,1,1,1
ring_lazy,60,15,10,1,20,26
ring_mid,40,1,7,1,1,1
travel_agent,25,25,1,1,25,1
